In [2]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
parquet_root = trec_root / "data/enwiki/parquet"
spark = get_spark()

categorylinks = spark.read.parquet((parquet_root / "categorylinks").as_posix())
nodes = spark.read.parquet((dataset_root / "graph/v1/nodes").as_posix())

categorylinks.printSchema()
nodes.printSchema()

root
 |-- cl_from: integer (nullable = true)
 |-- cl_to: string (nullable = true)
 |-- cl_sortkey: string (nullable = true)
 |-- cl_timestamp: string (nullable = true)
 |-- cl_sortkey_prefix: string (nullable = true)
 |-- cl_collation: string (nullable = true)
 |-- cl_type: string (nullable = true)
 |-- cl_collation_id: string (nullable = true)
 |-- cl_target_id: string (nullable = true)

root
 |-- page_id: integer (nullable = true)
 |-- wikidata_id: string (nullable = true)
 |-- node_id: integer (nullable = true)



In [4]:
nodes.count(), categorylinks.count()

(6336917, 205019381)

In [6]:
# let's see how many categories there are
pagecat = nodes.join(
    categorylinks.select(
        F.col("cl_from").alias("node_id"), F.col("cl_target_id").alias("category_id")
    ).distinct(),
    on="node_id",
    how="inner",
).cache()
pagecat.count()

23015923

In [ ]:
(
    pagecat.select("node_id").distinct().count(),
    pagecat.select("category_id").distinct().count(),
)

(2236100, 1105283)